# Lab 4 – Your project from the inside: `loader.py`, your `stats.py`, your `main.ipynb`
**Introduction to Python Programming for Data Science · AGH**

### Today
In Lab 2 you got a ready `loader.py` and only filled in your settings. Now you know loops and functions (lectures 3–4),
so today you **look inside** it, write its most important functions **yourself**, then build your **own** module `stats.py`
and **write** your `main.ipynb` yourself.

| Part | What you do | Data |
|---|---|---|
| 1 · Start | Setup – connects to your project and **checks** it | – |
| 2 · Your `loader.py` from the inside | read it function by function, then write 5 of them yourself | 🧪 test data |
| 3 · One recursion | lecture 4 | 🧪 test data |
| 4 · Your `stats.py` | 4 functions for everyone + **at least 2 of your choice** – then **you** put them into `src/stats.py` | 🧪 test data → your file |
| 5 · Your `main.ipynb` | open it the right way, **write** the analysis section yourself, Run all | **your data** |
| 6 · Result check | one cell – it also runs your `main.ipynb` from top to bottom | your data |

**🧪 Test data** = 800 Pokémon. They are **not your data** – only for practice, the same for everybody.
**This notebook never changes your project by itself.** You change your files. A file is saved only by a cell that says so (the result report at the end).
Not finished today? Finish it at home – every step is explained.

### How to work
Top to bottom · ▶ or **Shift + Enter** · **🔵** = you write something, replace `...` · 💡 hints: try first, then open ·
📌 = where to look in the documentation · ✅ / ❌ after every task.

## 1 · Start
| Google Colab | VS Code / PyCharm |
|---|---|
| 1. **File → Save a copy in Drive** – work in the copy | 1. Put this notebook into the **`notebooks/`** folder of your project and open it |
| 2. Run Setup (▶), click **Allow** for Google Drive | 2. Kernel **`.venv`** (as in Lab 3) · Run Setup – it finds your project one folder up |

**What Setup does:** loads the 🧪 test data (only into memory) · finds your project from Lab 2 · **checks** `src/loader.py`, `src/stats.py`, `main.ipynb`. It does **not** change any file.

**You should see:** `✅ TEST DATA: 800 Pokémon` and either `✅ Your project is ready for parts 4–6`
or `❌ …` – **what** is wrong and **which step of Lab 2** fixes it. Parts 2 and 3 work anyway. Fix it, then run Setup again.

In [1]:
# @title ⚙️ Setup Lab 4 – run this cell first (click ▶ or Shift+Enter)
# @markdown **Google Colab:** the name of your project folder on Google Drive (as in Lab 2):
PROJECT_NAME = "PyDS_project"  # @param {type:"string"}
# @markdown **VS Code / PyCharm:** the path to your project folder ("." = the folder of this notebook):
PROJECT_FOLDER = "."  # @param {type:"string"}

import csv, io, re, sys, json, importlib, urllib.request, types
from pathlib import Path
sys.dont_write_bytecode = True        # nothing is written into your project (no __pycache__)

# ---------- 1. where am I? ----------
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_NAME
    print("✅ running in Google Colab")
else:
    PROJECT = Path(PROJECT_FOLDER).resolve()
    if not (PROJECT / "src").exists():           # the notebook sits in notebooks/
        PROJECT = next((up for up in PROJECT.parents[:2] if (up / "src").exists()), PROJECT)
    print("✅ running on your computer (VS Code / PyCharm / Jupyter)")

# ---------- 2. TEST DATA: 800 Pokémon – only in memory, never saved ----------
DATA_URL = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"
_text = None
for place in [Path("pokemon.csv"), PROJECT / "notebooks" / "Lab03_local" / "pokemon.csv"]:
    if place.exists():
        _text = place.read_text(encoding="utf-8")
        break
if _text is None:
    try:
        _text = urllib.request.urlopen(DATA_URL, timeout=30).read().decode("utf-8")
    except Exception as error:
        print("❌ Could not download the test data:", error)
        print("   Take pokemon.csv from MS Teams and put it next to this notebook (Colab: 📁 → Upload), run again.")
RAW_TEST = list(csv.DictReader(io.StringIO(_text or "")))      # 🧪 like load_csv: every value is TEXT
TEST_TYPES = {"Name": str, "Type 1": str, "Type 2": str, "HP": int, "Attack": int, "Speed": int, "Legendary": bool}
if RAW_TEST:
    print(f"✅ TEST DATA: {len(RAW_TEST)} Pokémon (for practice – not your data)")

# ---------- 3. the loader of Lab 2 as a reference (in memory) – your file is not touched ----------
_ref = {"__file__": "/tmp/loader_reference.py", "__name__": "loader_reference"}
exec('"""loader.py – Lab 2: get the raw data into Python.\n\nWHAT GOES HERE\n    Everything needed to turn the raw data file into Python data structures:\n    reading the file, choosing the columns you need and converting text to\n    numbers / True-False.\n\n    PART A – ready-made helpers. Do not change them. From Lab 3 on you will\n             write functions like these yourself.\n    PART B – YOUR PROJECT SETTINGS. Fill them in during Lab 2: the name of\n             your data file and the columns you use, with their types.\n\nEXAMPLE (reference project – basketball shots)\n    DATA_FILE_NAME = "nba_shots_2024_25.csv"\n    COLUMN_TYPES = {"PLAYER_NAME": str, "SHOT_MADE": bool, "SHOT_DISTANCE": int}\n"""\nimport csv\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Folders of the project (this file lives in <project>/src/)\n# ---------------------------------------------------------------------------\nPROJECT_DIR = Path(__file__).resolve().parent.parent\nRAW_DIR = PROJECT_DIR / "data" / "raw"\nPROCESSED_DIR = PROJECT_DIR / "data" / "processed"\n\n# ===========================================================================\n# PART A – ready-made helpers (do not change)\n# ===========================================================================\n_ENCODINGS = ("utf-8-sig", "cp1250", "latin-1")\n_MISSING = {"", "na", "n/a", "nan", "none", "null", "-", "?"}\n\n\ndef _detect_delimiter(header_line):\n    """Pick the separator that splits the header line into the most columns."""\n    candidates = [",", ";", "\\t", "|"]\n    counts = {sep: header_line.count(sep) for sep in candidates}\n    best = max(counts, key=counts.get)\n    return best if counts[best] > 0 else ","\n\n\ndef load_csv(path, delimiter=None, encoding=None):\n    """Read a CSV file and return a list of rows (one dictionary per row).\n\n    All values are returned as text (str). Column names are stripped of spaces.\n    path      -- file name inside data/raw/ (e.g. "my_data.csv") or a full path\n    delimiter -- None = detect automatically (",", ";", tab or "|")\n    encoding  -- None = try UTF-8, then cp1250 (Polish Excel), then latin-1\n    """\n    path = Path(path)\n    if not path.is_absolute() and not path.exists():\n        path = RAW_DIR / path\n    if not path.exists():\n        available = sorted(p.name for p in RAW_DIR.glob("*") if p.name != "README.md") if RAW_DIR.exists() else []\n        raise FileNotFoundError(\n            f"File not found: {path}\\nFiles in data/raw/: {available}\\n"\n            "Check the name (capital letters, .csv at the end)."\n        )\n    if path.suffix.lower() in (".gsheet", ".xlsx", ".xls"):\n        raise ValueError(\n            f"{path.name} is not a CSV file. Open it in Google Sheets / Excel and use "\n            "File -> Download -> Comma-separated values (.csv), then upload the .csv."\n        )\n\n    encodings = (encoding,) if encoding else _ENCODINGS\n    for enc in encodings:\n        try:\n            text = path.read_text(encoding=enc)\n            break\n        except UnicodeDecodeError:\n            continue\n    else:\n        raise UnicodeDecodeError("csv", b"", 0, 1, f"cannot decode {path.name}")\n\n    lines = text.splitlines()\n    if not lines:\n        return []\n    sep = delimiter or _detect_delimiter(lines[0])\n    reader = csv.DictReader(lines, delimiter=sep)\n    reader.fieldnames = [name.strip() for name in reader.fieldnames]\n    rows = []\n    for row in reader:\n        if any(value not in (None, "") for value in row.values()):\n            rows.append({key: (value or "").strip() for key, value in row.items() if key})\n    return rows\n\n\ndef save_csv(rows, path):\n    """Save a list of dictionaries as a CSV file (creates the folder if needed)."""\n    path = Path(path)\n    if not path.is_absolute():\n        path = RAW_DIR / path\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))\n        writer.writeheader()\n        writer.writerows(rows)\n    return path\n\n\ndef to_number(text):\n    """\'1,5\' -> 1.5 · \'26\' -> 26.0 · \'\' or \'NA\' -> None · \'abc\' -> None"""\n    if text is None:\n        return None\n    if isinstance(text, (int, float)):\n        return float(text)\n    text = str(text).strip().replace(" ", "").replace(",", ".")\n    if text.lower() in _MISSING:\n        return None\n    try:\n        return float(text)\n    except ValueError:\n        return None\n\n\ndef to_bool(text):\n    """\'True\', \'yes\', \'1\', \'t\' -> True · \'False\', \'no\', \'0\', \'f\' -> False · other -> None"""\n    value = str(text).strip().lower()\n    if value in ("true", "t", "yes", "y", "1"):\n        return True\n    if value in ("false", "f", "no", "n", "0"):\n        return False\n    return None\n\n\ndef _convert(value, kind):\n    if kind is bool:\n        return to_bool(value)\n    if kind is int:\n        number = to_number(value)\n        return None if number is None else int(number)\n    if kind is float:\n        return to_number(value)\n    text = str(value).strip()\n    return None if text.lower() in _MISSING else text\n\n\ndef convert_rows(rows, column_types):\n    """Keep only the columns from column_types and convert their values.\n\n    column_types -- dictionary {column name: type}, type is str, int, float or bool\n    Values that cannot be converted become None.\n    """\n    missing = [name for name in column_types if rows and name not in rows[0]]\n    if missing:\n        raise KeyError(f"Columns not in the data: {missing}. Available: {list(rows[0])}")\n    return [{name: _convert(row[name], kind) for name, kind in column_types.items()} for row in rows]\n\n\ndef get_column(rows, name, skip_missing=True):\n    """Return the values of one column as a list (None values skipped by default)."""\n    values = [row[name] for row in rows]\n    return [v for v in values if v is not None] if skip_missing else values\n\n\n# ===========================================================================\n# PART B – YOUR PROJECT SETTINGS (fill in during Lab 2)\n# ===========================================================================\nDATA_FILE_NAME = "my_data.csv"      # the exact name of your file in data/raw/\n\n# The columns you need for your question and their types (str, int, float, bool).\n# Explain WHY you chose each column in docs/data_dictionary.md.\n# Example (basketball): {"PLAYER_NAME": str, "SHOT_MADE": bool, "SHOT_DISTANCE": int}\nCOLUMN_TYPES = {\n}\n\n\ndef load_project_data():\n    """Read the project data file and return converted rows."""\n    return convert_rows(load_csv(DATA_FILE_NAME), COLUMN_TYPES)\n', _ref)
L = types.SimpleNamespace(**_ref)            # L.to_bool, L.convert_rows, … = the same code as PART A of your loader.py
pokemon_rows = L.convert_rows(RAW_TEST, TEST_TYPES)             # 🧪 test data with the right types

# ---------- 4. helpers ----------
RESULTS = {}

def check(label, test, hint=""):
    try:
        ok = bool(test())
    except Exception as error:
        ok = False
        hint = hint or str(error)
    RESULTS[label] = ok
    print(("✅ " if ok else "❌ ") + label + ("" if ok or not hint else "   → hint: " + hint))

def fresh(module):
    """Import src/<module>.py of your project again – always the newest version."""
    for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
        del sys.modules[name]
    while str(PROJECT) in sys.path:
        sys.path.remove(str(PROJECT))
    sys.path.insert(0, str(PROJECT))         # always YOUR project first
    importlib.invalidate_caches()
    return importlib.import_module("src." + module)

def notebook_cells(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))["cells"] or None
    except Exception:
        return None


def run_main():
    """Runs every code cell of your main.ipynb, top to bottom. Returns (ok, message, the stats functions it imports)."""
    import os, contextlib
    cells = notebook_cells(PROJECT / "main.ipynb") or []
    sources = ["".join(c["source"]) for c in cells if c["cell_type"] == "code"]
    used = sorted({f.strip() for s in sources for part in re.findall(r"from src\.stats import ([\w ,]+)", s) for f in part.split(",") if f.strip()})
    space, here, old_path = {"__name__": "__main__"}, os.getcwd(), list(sys.path)
    if globals().get("get_ipython"):
        space["get_ipython"] = get_ipython
    os.chdir(PROJECT)
    try:
        for i, src in enumerate(sources):
            if not src.strip():
                continue
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    try:
                        code_text = get_ipython().transform_cell(src)
                    except Exception:
                        code_text = src
                    exec(compile(code_text, f"main.ipynb cell {i}", "exec"), space)
            except Exception as error:
                return False, f"code cell {i + 1} ({src.strip().splitlines()[0][:45]}…): {type(error).__name__}: {error}", used
    finally:
        os.chdir(here)
        sys.path[:] = old_path               # main.ipynb must not change this notebook
    return bool(sources), "ok", used

MENU_TESTS = {                                   # the menu of part 4.5: (call, expected result)
    "std": (lambda: std([2, 4, 4, 4, 5, 5, 7, 9]), 2.14),
    "share": (lambda: share([True, False, True, True]), 0.75),
    "filter_rows": (lambda: [r["name"] for r in filter_rows(MINI, lambda r: r["type1"] == "Fire")], ["Charmander", "Charmeleon"]),
    "top": (lambda: top({"a": 1, "b": 3, "c": 2}, n=2), [("b", 3), ("c", 2)]),
    "min_max": (lambda: min_max([3, 1, 2]), (1, 3)),
    "summary_by": (lambda: summary_by(MINI, "type1", "attack"), [{"group": "Fire", "count": 2, "mean": 58.0}, {"group": "Grass", "count": 2, "mean": 55.5}, {"group": "Water", "count": 1, "mean": 48.0}]),
}

def menu_working(module=None):
    """Which menu functions work: (working, written but wrong). module=None: this notebook, else your src/stats.py."""
    ok, wrong = [], []
    for name, (call, expected) in MENU_TESTS.items():
        if module is not None and not hasattr(module, name):
            continue
        saved = globals().get(name)
        if module is not None:
            globals()[name] = getattr(module, name)
        try:
            value = call()
            if value is not None:
                (ok if value == expected else wrong).append(name)
        except Exception:
            pass                                 # not written yet (still ...)
        finally:
            if module is not None:
                globals()[name] = saved
    return ok, wrong

class ProjectNotReady(Exception):
    pass

def require_project():
    if not PROJECT_OK:
        raise ProjectNotReady("❌ This part needs your project from Lab 2. Fix this first, then run Setup again:\n   "
                              + "\n   ".join(PROBLEMS))

def _friendly_error(shell, etype, value, tb, tb_offset=None):
    if etype is ProjectNotReady:
        print(value)
    elif "ellipsis" in str(value) or "NoneType" in str(value):
        print("❌ Something is not written yet (a `...` or a function without return).")
        print("   Python said:", etype.__name__ + ":", value)
    elif etype is NameError:
        print("❌", value, "→ run the cells above first.")
    else:
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)

try:
    get_ipython().set_custom_exc((ProjectNotReady, TypeError, NameError, AttributeError), _friendly_error)
except Exception:
    pass

# ---------- 5. your project from Lab 2 – only CHECKED, nothing is changed ----------
PROBLEMS, NOTES = [], []
loader, rows = None, None
RAW_DIR = PROJECT / "data" / "raw"
loader_file = PROJECT / "src" / "loader.py"
print("\nYour project:", PROJECT)
if not PROJECT.exists():
    PROBLEMS.append(f"There is no folder {PROJECT}. Colab: write the name of your project folder in PROJECT_NAME (as in Lab 2). VS Code / PyCharm: set PROJECT_FOLDER.")
elif not loader_file.exists():
    PROBLEMS.append("src/loader.py is missing → Lab 2: run its Setup cell, then steps 3.1–3.4.")
else:
    if "PART B" not in loader_file.read_text(encoding="utf-8"):
        PROBLEMS.append("src/loader.py is still the Lab 1 version (no PART B) → do Lab 2: its Setup cell, then steps 3.1–3.4.")
    else:
        if str(PROJECT) not in sys.path:
            sys.path.insert(0, str(PROJECT))
        try:
            loader = fresh("loader")
        except Exception as error:
            PROBLEMS.append(f"src/loader.py has an error: {type(error).__name__}: {error} → open it and fix the line Python shows.")
        if loader is not None:
            name = getattr(loader, "DATA_FILE_NAME", "")
            col_types = getattr(loader, "COLUMN_TYPES", {})
            files = sorted(p.name for p in RAW_DIR.glob("*.csv")) if RAW_DIR.exists() else []
            if not name or name == "my_data.csv":
                PROBLEMS.append('src/loader.py, PART B: DATA_FILE_NAME is still "my_data.csv" → write the name of YOUR file (Lab 2, step 3.3).')
            elif name.startswith("practice_"):
                PROBLEMS.append(f'src/loader.py, PART B: "{name}" is the practice file of Lab 2, not your data → put YOUR file into data/raw/ (Lab 2, step 3.1) and write its name (step 3.3).')
            elif name not in files:
                PROBLEMS.append(f'data/raw/: there is no file "{name}" (files there: {files}) → upload it (Lab 2, step 3.1) or correct DATA_FILE_NAME (step 3.3).')
            if not col_types:
                PROBLEMS.append("src/loader.py, PART B: COLUMN_TYPES is empty → write your columns and their types (Lab 2, step 3.3).")
            if not PROBLEMS:
                try:
                    rows = loader.load_project_data()
                    print(f"✅ src/loader.py loads your data: {name} – {len(rows)} rows, columns {list(col_types)}")
                except Exception as error:
                    PROBLEMS.append(f"load_project_data() does not work: {type(error).__name__}: {error} → check the column names in COLUMN_TYPES (Lab 2, step 3.4).")
            if not PROBLEMS and not any(k is str for k in col_types.values()):
                NOTES.append("COLUMN_TYPES has no text column (str) – for count_by / group_stat you need one category column. Add one in PART B if your file has it.")
            if not PROBLEMS and not any(k in (int, float) for k in col_types.values()):
                NOTES.append("COLUMN_TYPES has no number column (int / float) – you need one for mean / median. Add one in PART B.")
    if not (PROJECT / "src" / "stats.py").exists():
        PROBLEMS.append("src/stats.py is missing → 📁 right-click the folder src → New file → stats.py.")
    main_file = PROJECT / "main.ipynb"
    if not main_file.exists():
        PROBLEMS.append("main.ipynb is missing → run the Setup cell of Lab 2 again, it creates it.")
    elif notebook_cells(main_file) is None:
        PROBLEMS.append("main.ipynb is empty or broken → rename it to main_old.ipynb and run the Setup cell of Lab 2 again, it creates a new one.")

PROJECT_OK = not PROBLEMS
for note in NOTES:
    print("⚠️ ", note)
if PROJECT_OK:
    print("✅ Your project is ready for parts 4–6")
else:
    print("❌ Your project is not ready for parts 4–6 – parts 2 and 3 work anyway (test data). Fix this, then run Setup again:")
    for problem in PROBLEMS:
        print("   ❌", problem)

✅ running on your computer (VS Code / PyCharm / Jupyter)
✅ TEST DATA: 800 Pokémon (for practice – not your data)

Your project: C:\AGH sem7\pythonindatascience\PyDataProject
✅ src/loader.py loads your data: Student_Depression_Dataset.csv – 27901 rows, columns ['Age', 'Gender', 'CGPA', 'Sleep Duration', 'Financial Stress', 'Depression']
✅ Your project is ready for parts 4–6


## 2 · Your `loader.py` from the inside · 🧪 test data
This is the path of your data every time you click **Run all** in `main.ipynb`:

```
data/raw/your_file.csv ──load_csv()──► raw: rows of TEXT ──convert_rows()──► rows: the right TYPES ──get_column()──► one list of values
                                                          (uses _convert → to_number / to_bool)
```

You open `src/loader.py` (📁 → your project → `src` → double-click `loader.py`) and go through it **function by function**.
For every function: **what it is for** · **the code, explained line by line** · **predict** · and for 5 of them **write it yourself**.
Your own version is compared with the loader (`L.…` = the same code as PART A of your file) – it must give the same result.


📌 **Under your nose** – exactly where to look:
- functions, default values → [docs.python.org/3/tutorial/controlflow.html#defining-functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions)

### 2.1 `load_csv()` – read the file (you only read and understand it)
**What it is for:** it opens your CSV file and gives you back a **list of rows**. Every row is a **dictionary**: *column name → value*.
⚠️ **Every value is text** (`str`) – even the numbers. `"45"` is text, not the number 45. That is why the next functions exist.

**What happens, in plain words:**
1. **Find the file.** You give only a name like `"my_data.csv"` → the function looks for it in the folder `data/raw/`.
   No file there? → a clear error that lists the files that **are** in `data/raw/`.
2. **Read the text.** It tries 3 encodings one after another (UTF-8, then the Polish Excel one, then latin-1) until the letters look right.
3. **Find the separator.** Some files use `,`, some `;` (Polish Excel), some a tab. It takes the one that splits the first line into the most columns.
4. **Make dictionaries.** The `csv` module turns every line into a dictionary – the column names come from the first line.
5. **Clean up.** Completely empty lines are skipped, spaces around values are removed.

**Example.** The file
```
Name,HP,Legendary
Bulbasaur,45,False
```
becomes
```python
[{"Name": "Bulbasaur", "HP": "45", "Legendary": "False"}]     # "45" and "False" are TEXT
```

The code is in your `src/loader.py` (📁 → your project → `src` → double-click `loader.py`) – you may look, but you do **not** write it today:
reading files and `try/except` come in lectures 6–7. Then you write your own `load_csv` (Lab 7).

🧪 `RAW_TEST` is exactly what `load_csv` gives for the Pokémon file.
**Predict** before you run: what is the **type** of `RAW_TEST[0]["HP"]`? How many rows are there?


📌 **Under your nose** – exactly where to look:
- csv.DictReader – how a line becomes a dictionary → [docs.python.org/3/library/csv.html#csv.DictReader](https://docs.python.org/3/library/csv.html#csv.DictReader)

In [2]:
my_type = "str"            # 🔵 the type of RAW_TEST[0]["HP"] as text: "str", "int", "float" or "bool"
my_rows = 800             # 🔵 how many rows?

print(RAW_TEST[0])                                       # the first row: a dictionary
print(type(RAW_TEST[0]["HP"]).__name__, len(RAW_TEST))   # its type, and the number of rows


# ---------- check (do not change) ----------
check("2.1 the type", lambda: my_type == "str", "load_csv gives only text")
check("2.1 rows", lambda: my_rows == len(RAW_TEST))

{'#': '1', 'Name': 'Bulbasaur', 'Type 1': 'Grass', 'Type 2': 'Poison', 'HP': '45', 'Attack': '49', 'Defense': '49', 'Sp. Atk': '65', 'Sp. Def': '65', 'Speed': '45', 'Generation': '1', 'Legendary': 'FALSE'}
str 800
✅ 2.1 the type
✅ 2.1 rows


### 2.2 `to_bool()` – text → True / False ✍️ your own, better version
**What it is for:** a yes/no column is read as **text**: `"True"`, `"False"`, `" yes "`, `"0"` …
**The trap:** Python's `bool("False")` gives `True` – for Python **every non-empty text is True**. So we cannot just write `bool(text)`.

**How the loader's `to_bool` does it:**
```python
def to_bool(text):
    value = str(text).strip().lower()             # make the text "clean"
    if value in ("true", "t", "yes", "y", "1"):   # is it one of the YES words?
        return True
    if value in ("false", "f", "no", "n", "0"):   # is it one of the NO words?
        return False
    return None                                   # none of them -> we do not know
```
**Step by step with the text `" Yes "`:**
1. `str(text)` – make sure it is text → `" Yes "`
2. `.strip()` – remove the spaces at the start and the end → `"Yes"`
3. `.lower()` – small letters → `"yes"`
4. `"yes" in ("true", "t", "yes", "y", "1")` – is `"yes"` in this group of words? **Yes** → the function gives back `True` and **stops** (`return` ends the function).
With `"maybe"`: it is in neither group → the last line → `None`.

**🔵 Your task – one function for any data.** Real data says yes / no in many ways: a basketball shot is `"made"` / `"missed"`,
a customer is `"active"` / `"inactive"`, an exam is `"pass"` / `"fail"`. The loader only knows its own two groups of words – they are written **inside** the function.

Write `my_to_bool(text, yes=..., no=...)` where the two groups of words are **parameters with default values**:
- the default values are the loader's words → without extra arguments it works **exactly like the loader**,
- with your own words it understands your data: `my_to_bool("Missed", yes=("made",), no=("missed",))` → `False`.

**Think:** where in the loader's code are the two groups of words? What do they become in your version – and what do you write in the two `if`s instead?
(Default values: like `skip_missing=True` in `get_column`, lecture 4.)
**You should see:** `True True False None` and three ✅.
<details><summary>💡 Hint 1 (click)</summary>

The first line: `def my_to_bool(text, yes=("true", "t", "yes", "y", "1"), no=("false", "f", "no", "n", "0")):`

</details>
<details><summary>💡 Hint 2 (click)</summary>

Inside, the groups are now variables: `if value in yes: return True` · `if value in no: return False` · `return None`. Write your own words in small letters – the text is `.lower()`-ed.

</details>

📌 **Under your nose** – exactly where to look:
- str.strip and str.lower → [docs.python.org/3/library/stdtypes.html#str.strip](https://docs.python.org/3/library/stdtypes.html#str.strip)
- `in` – is a value in a group? → [docs.python.org/3/reference/expressions.html#membership-test-operations](https://docs.python.org/3/reference/expressions.html#membership-test-operations)

In [3]:
print(bool("False"))              # 👀 the trap: a non-empty text is always True

# ✍️ your code – my_to_bool() with the groups of words as parameters
def my_to_bool(text, yes=("true","t", "yes", "1"), no=("false", "f", "no", "0")):
    value = str(text).strip().lower()
    if value in yes:
        return True
    if value in no:
        return False
    return None
print(my_to_bool(" TRUE "), my_to_bool("Made", yes=("made",), no=("missed",)), my_to_bool("missed", yes=("made",), no=("missed",)), my_to_bool("maybe"))


# ---------- check (do not change) ----------
english = ["True", "False", " yes ", "NO", "1", "0", "maybe", ""]
check("2.2 without extra arguments = the loader", lambda: [my_to_bool(s) for s in english] == [L.to_bool(s) for s in english],
      "the default values must be the loader's words")
check("2.2 your own words", lambda: [my_to_bool(s, yes=("active",), no=("inactive",)) for s in [" Active", "INACTIVE", "yes"]] == [True, False, None],
      "inside: if value in yes / if value in no")
check("2.2 spaces, capital letters", lambda: my_to_bool("  Yes  ") is True, ".strip().lower() first")

True
True True False None
✅ 2.2 without extra arguments = the loader
✅ 2.2 your own words
✅ 2.2 spaces, capital letters


### 2.3 `to_number()` – text → number ✍️ write it yourself (simple version)
**What it is for:** numbers in a CSV file are **text**: `"26"`, `"1,5"` (with a Polish comma!), `""` (an empty cell), `"NA"` (= not available).
You cannot calculate with text: `"26" + "1"` gives `"261"`, not 27. So every number column goes through `to_number`.

**How it works, in plain words:**
1. Nothing (`None`)? → give back nothing (`None`).
2. Clean the text: remove spaces, change the Polish comma into a dot: `" 1,5 "` → `"1.5"`.
3. Is it one of the "missing" words (`""`, `"na"`, `"n/a"`, `"nan"`, `"none"`, `"null"`, `"-"`, `"?"`)? → `None`.
4. Otherwise: `float("1.5")` → the number `1.5`.

**Step by step with `" 1,5 "`:** `str()` → `" 1,5 "` · `.strip()` → `"1,5"` · `.replace(",", ".")` → `"1.5"` · not missing · `float("1.5")` → **1.5**
**And with `"NA"`:** cleaned → `"NA"` · `.lower()` → `"na"` → it is a missing word → **None**

In your loader there is one more thing: `try: … except ValueError: …`. It means *"try to make a number – if it does not work (e.g. `"abc"`), give `None` instead of an error"*.
That is lecture 6 – today you write the version **without** it.

**🔵 Your task – numbers with units.** Real files often have extra characters next to a number: a unit `"25 °C"`, a currency `"1 299,99 zł"`,
a percent `"12,5%"`. The loader gives `None` for all of them – your data would lose these values!
Write `my_to_number(text)` that **keeps only the characters that belong to a number** – digits `0`–`9`, the minus `-` and the decimal dot:
1. `None` → `None`.
2. Change the comma into a dot: `"1 299,99 zł"` → `"1 299.99 zł"`.
3. **A loop over the characters** of the text: build a new text `clean` that gets only the digits, `-` and `.` → `"1299.99"`.
   `ch.isdigit()` tells you if the character `ch` is a digit.
4. Nothing useful left (`""`, `"-"` or `"."`)? → `None`. (This also makes `"NA"`, `"n/a"`, `"?"` → `None`.)
5. Otherwise `float(clean)`.

**You should see:** `26.0 1299.99 25.0 -3.5 None` and three ✅.
<details><summary>💡 Hint 1 (click)</summary>

Step 3: start with `clean = ""` · `for ch in text:` · `if ch.isdigit() or ch in "-.": clean = clean + ch`

</details>
<details><summary>💡 Hint 2 (click)</summary>

Step 4–5: `if clean in ("", "-", "."): return None` · `return float(clean)`. Do not forget `text = str(text)` at the start – `None` was handled in step 1.

</details>

📌 **Under your nose** – exactly where to look:
- str.isdigit – is this character a digit? → [docs.python.org/3/library/stdtypes.html#str.isdigit](https://docs.python.org/3/library/stdtypes.html#str.isdigit)
- str.replace – change one piece of text into another → [docs.python.org/3/library/stdtypes.html#str.replace](https://docs.python.org/3/library/stdtypes.html#str.replace)
- float() – text into a number → [docs.python.org/3/library/functions.html#float](https://docs.python.org/3/library/functions.html#float)

In [4]:
# ✍️ your code – my_to_number()
def my_to_number(text):
    if text is None:
          return None
    
    text = str(text).strip().replace(" ", "").replace(",", ".")
    clean = ""
    for ch in text:
          if ch.isdigit() or ch in "-.":
              clean += ch
    if clean in ("", "-", "."):
        return None
    return float(clean)
print(my_to_number("26"), my_to_number("1 299,99 zł"), my_to_number("25 °C"), my_to_number("-3,5"), my_to_number("NA"))


# ---------- check (do not change) ----------
samples = ["26", "1,5", " 3.25 ", "1 234,5", "", "NA", "n/a", "-", "?", None]
check("2.3 my_to_number – units, currency, percent", lambda: (my_to_number("1 299,99 zł"), my_to_number("25 °C"), my_to_number("12,5%")) == (1299.99, 25.0, 12.5),
      "the loop must keep only digits, - and .")
check("2.3 my_to_number – minus", lambda: my_to_number("-3,5") == -3.5, "keep the - too")
check("2.3 everything the loader understands", lambda: [my_to_number(s) for s in samples] == [L.to_number(s) for s in samples])

26.0 1299.99 25.0 -3.5 None
✅ 2.3 my_to_number – units, currency, percent
✅ 2.3 my_to_number – minus
✅ 2.3 everything the loader understands


### 2.4 `_convert()` – choose the right conversion ✍️ write it yourself
**What it is for:** in Lab 2 you wrote in `COLUMN_TYPES` a **type** for every column, e.g. `"HP": int`, `"Legendary": bool`.
This function looks at that type and chooses **which** conversion to use. It is the "switch" between your functions.

**How it works, in plain words:** you give it a value (text) and a type.
- type `bool` → use `to_bool` (2.2)
- type `int` → use `to_number`, then cut the decimals: `26.0` → `26` (but if it is `None`, keep `None` – `int(None)` would be an error)
- type `float` → use `to_number` (2.3)
- type `str` → only clean the text; an empty text → `None`

**Step by step:** `_convert("26", int)`: the type is `int` → `to_number("26")` → `26.0` → it is not `None` → `int(26.0)` → **26**
`_convert(" Pikachu ", str)`: not bool, not int, not float → it is text → `.strip()` → **"Pikachu"**

**New:** a type is a value too, so you can check it with `is`: `if kind is int:` means *"if the type is int"*.
The `_` at the start of the name means *"a helper, used only inside the module"*.

**🔵 Your task – one thing better.** `int(26.7)` simply **cuts** the decimals → `26`. That is not correct: 26.7 is closer to **27**.
Write `my_convert(value, kind)` – four `if`s, one per type, with **your** `my_to_bool` and `my_to_number` – but for `int` it must **round** to the nearest whole number.
Which built-in function rounds? (You used it with `digits` in Lab 3.)
**You should see:** `True 27 1.5 Pikachu None` and three ✅.
<details><summary>💡 Hint 1 (click)</summary>

`if kind is bool: return my_to_bool(value)` – and the same for `float` with `my_to_number`. For `str`: `text = str(value).strip()` · `return None if text == "" else text`

</details>
<details><summary>💡 Hint 2 (click)</summary>

int: `number = my_to_number(value)` · `return None if number is None else round(number)` – `round()` without digits gives a whole number

</details>

In [5]:
# ✍️ your code – my_convert()
def my_convert(text, type0):
    match type0.__name__:
        case "bool":
            return my_to_bool(text)
        case "float":
            return my_to_number(text)
        case "int":
            nb = my_to_number(text)
            return None if nb is None else round(nb)
        case _:
            text2 = str(text).strip()
            return None if text2 == "" else text2
            
              # 🔵 delete this line and write your function here
print(my_convert("True", bool), my_convert("26,7 kg", int), my_convert("1,5", float), my_convert(" Pikachu ", str), my_convert("", str))


# ---------- check (do not change) ----------
check("2.4 my_convert – every type", lambda: (my_convert("True", bool), my_convert("26", int), my_convert("1,5", float), my_convert(" Pikachu ", str)) == (True, 26, 1.5, "Pikachu"))
check("2.4 my_convert – int is rounded", lambda: my_convert("26,7", int) == 27 and type(my_convert("26,7", int)) is int, "round(number)")
check("2.4 my_convert – empty → None", lambda: my_convert("", str) is None and my_convert("", int) is None)

True 27 1.5 Pikachu None
✅ 2.4 my_convert – every type
✅ 2.4 my_convert – int is rounded
✅ 2.4 my_convert – empty → None


### 2.5 `convert_rows()` – all rows, only your columns ✍️ write it yourself
**What it is for:** this is the line `rows = convert_rows(raw, COLUMN_TYPES)` in your `main.ipynb`.
It goes through **every row** of the file and makes a **new row** that has **only the columns of `COLUMN_TYPES`**, every value converted with `_convert`.

**How it works – two loops, one inside the other:**
```python
new_rows = []                                   # 1. an empty list for the new rows
for row in rows:                                # 2. take the rows one by one
    new_row = {}                                #    3. an empty dictionary for this one row
    for name, kind in column_types.items():     #    4. take your columns one by one: name and type
        new_row[name] = _convert(row[name], kind)   #   5. the converted value goes into the new row
    new_rows.append(new_row)                    #    6. the finished row goes into the list
return new_rows                                 # 7. after ALL rows: give back the list
```
**Step by step with one row** `{"Name": "Bulbasaur", "HP": "45", "Speed": "45"}` and `column_types = {"Name": str, "HP": int}`:
- step 3: `new_row = {}`
- step 4, first column: `name = "Name"`, `kind = str` → step 5: `new_row = {"Name": "Bulbasaur"}`
- step 4, second column: `name = "HP"`, `kind = int` → step 5: `new_row = {"Name": "Bulbasaur", "HP": 45}`
- step 6: this row goes into `new_rows`. `"Speed"` is not in `column_types` → it is left out.

In the loader the same is written shorter, in one line with comprehensions (lecture 3) – it does exactly the same:
`[{name: _convert(row[name], kind) for name, kind in column_types.items()} for row in rows]`

**🔵 Your task – two functions.**
1. `my_convert_rows(rows, column_types)` – the two loops above, with **your** `my_convert`.
2. **New:** `missing_report(rows)` – the first check of **data quality**: for every column, how many values are `None` after the conversion?
   A dictionary like `{"Name": 0, "Type 2": 386, "HP": 0, …}`. Nobody shows you this code – you build it from what you know:
   - you need **two loops** again: over the rows, and inside over `row.items()` (name and value),
   - every column name must be in the dictionary (also with 0) – like in `count_by` from Lab 3: *not a key yet → put it there with 0*,
   - if the value `is None` → add 1.

You will need this report in Lab 6 (cleaning the data).
🧪 **You should see:** the first row before (only text) and after (numbers as numbers), then the report, and three ✅.
<details><summary>💡 Hint 1 (click)</summary>

my_convert_rows: follow the 7 steps above – only change `_convert` into `my_convert`.

</details>
<details><summary>💡 Hint 2 (click)</summary>

missing_report: `report = {}` · `for row in rows:` · `for name, value in row.items():` · `if name not in report: report[name] = 0` · `if value is None: report[name] = report[name] + 1` · `return report`

</details>

📌 **Under your nose** – exactly where to look:
- a loop over a dictionary: `.items()` → [docs.python.org/3/tutorial/datastructures.html#looping-techniques](https://docs.python.org/3/tutorial/datastructures.html#looping-techniques)
- comprehensions (the short form) → [docs.python.org/3/tutorial/datastructures.html#list-comprehensions](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions)

In [6]:
# ✍️ your code – my_convert_rows() and missing_report()
def my_convert_rows(rows, column_types):
      new_rows = []
      for r in rows:
            new_row = {}
            for name, kind in column_types.items():
                  new_row[name] = my_convert(r.get(name), kind)
            new_rows.append(new_row)
      return new_rows

def missing_report(rows):
      report = {name: 0 for name in rows[0].keys()} if rows else {}
      for r in rows:
            for name, value in r.items():
                  if value is None:
                        report[name] += 1
      return report
my_rows_test = my_convert_rows(RAW_TEST, TEST_TYPES)          # 🧪 all 800 rows of the test data
print("before:", RAW_TEST[0])
print("after:    ", my_rows_test[0])
print("missing values:", missing_report(my_rows_test))


# ---------- check (do not change) ----------
check("2.5 only the columns of TEST_TYPES", lambda: list(my_rows_test[0]) == list(TEST_TYPES))
check("2.5 missing_report", lambda: missing_report(my_rows_test) == {"Name": 0, "Type 1": 0, "Type 2": 386, "HP": 0, "Attack": 0, "Speed": 0, "Legendary": 0},
      "every column must be there, also with 0")
check("2.5 my_convert_rows = L.convert_rows (800 rows)", lambda: my_rows_test == L.convert_rows(RAW_TEST, TEST_TYPES), "compare your first row with the loader's: L.convert_rows(RAW_TEST, TEST_TYPES)[0]")

before: {'#': '1', 'Name': 'Bulbasaur', 'Type 1': 'Grass', 'Type 2': 'Poison', 'HP': '45', 'Attack': '49', 'Defense': '49', 'Sp. Atk': '65', 'Sp. Def': '65', 'Speed': '45', 'Generation': '1', 'Legendary': 'FALSE'}
after:     {'Name': 'Bulbasaur', 'Type 1': 'Grass', 'Type 2': 'Poison', 'HP': 45, 'Attack': 49, 'Speed': 45, 'Legendary': False}
missing values: {'Name': 0, 'Type 1': 0, 'Type 2': 386, 'HP': 0, 'Attack': 0, 'Speed': 0, 'Legendary': 0}
✅ 2.5 only the columns of TEST_TYPES
✅ 2.5 missing_report
✅ 2.5 my_convert_rows = L.convert_rows (800 rows)


### 2.6 `get_column()` – one column as a list ✍️ write it yourself
**What it is for:** to calculate a mean or a maximum you need **one list of numbers** – not a list of rows.
`get_column(rows, "HP")` takes the value `"HP"` from every row: `[45, 60, 39, …]`.

**Missing values:** some cells are empty → after `convert_rows` they are `None`. `sum([45, None])` is an **error**.
So the function has a **default value**: `skip_missing=True`. If you do not say anything, the `None`s are left out.
If you want them, you say so: `get_column(rows, "HP", skip_missing=False)`.

**How it works:**
```python
values = [row[name] for row in rows]            # one value from every row (a comprehension)
if skip_missing:                                # the default: True
    return [v for v in values if v is not None] #   only the values that are not None
return values                                   # skip_missing=False: all values, also None
```
**Example:** 3 rows with HP `45`, `None`, `60` → `get_column(rows, "HP")` → `[45, 60]` · with `skip_missing=False` → `[45, None, 60]`

**🔵 Your task:** first **predict**: the column `"Type 2"` (many Pokémon have no second type) – how many values **with** and **without** skipping?
Then write your **better** version: `my_get_column(rows, name, skip_missing=True, unique=False)` – like the loader, plus one more default parameter:
`unique=True` → every value **only once**, **sorted** – e.g. all types of Pokémon: `['Bug', 'Dark', 'Dragon', …]`.
Which data structure keeps every value only once? (Lab 2!) And which function sorts?
**You should see:** `800 414 18` and three ✅.
<details><summary>💡 Hint 1 (click)</summary>

The prediction: without skipping you get one value per row. With skipping: fewer – run `len(L.get_column(pokemon_rows, "Type 2"))` if you are not sure.

</details>
<details><summary>💡 Hint 2 (click)</summary>

A `set` keeps every value once; `sorted()` makes a sorted list: `if unique: return sorted(set(values))` – after skipping the `None`s.

</details>

📌 **Under your nose** – exactly where to look:
- default values of parameters → [docs.python.org/3/tutorial/controlflow.html#default-argument-values](https://docs.python.org/3/tutorial/controlflow.html#default-argument-values)

In [7]:
my_guess_all = 800        # 🔵 len(my_get_column(pokemon_rows, "Type 2", skip_missing=False))
my_guess_skip = 414       # 🔵 len(my_get_column(pokemon_rows, "Type 2"))

# ✍️ your code – my_get_column()
def my_get_column(rows, name, skip_missing=True, unique=False):
      value = [r[name] for r in rows]
      if skip_missing:
            value = [v for v in value if v is not None]
      if unique:
            value = sorted(set(value))
      return value
print(len(my_get_column(pokemon_rows, "Type 2", skip_missing=False)), len(my_get_column(pokemon_rows, "Type 2")),
      len(my_get_column(pokemon_rows, "Type 1", unique=True)))


# ---------- check (do not change) ----------
check("2.6 your guesses", lambda: (my_guess_all, my_guess_skip) == (800, 414), "386 Pokémon have no Type 2")
check("2.6 my_get_column = L.get_column", lambda: my_get_column(pokemon_rows, "Type 2") == L.get_column(pokemon_rows, "Type 2")
      and my_get_column(pokemon_rows, "Type 2", skip_missing=False) == L.get_column(pokemon_rows, "Type 2", skip_missing=False))
check("2.6 unique=True", lambda: my_get_column(pokemon_rows, "Type 1", unique=True) == sorted(set(L.get_column(pokemon_rows, "Type 1"))), "sorted(set(values))")

800 414 18
✅ 2.6 your guesses
✅ 2.6 my_get_column = L.get_column
✅ 2.6 unique=True


### 2.7 `save_csv()` – write rows into a file (you only read and understand it)
**What it is for:** the opposite of `load_csv`: a list of dictionaries → a CSV file. You used it in Lab 2 (`krakow_week.csv`).

**What happens, in plain words:**
1. Only a name like `"test.csv"`? → the file goes into the folder `data/raw/`.
2. The folder does not exist? → it is made.
3. The file is opened **for writing** (`"w"`) – an old file with the same name is replaced.
4. **Line 1 = the header:** the column names – taken from the keys of the first dictionary.
5. Then **one line per dictionary**.

**Example:** `[{"Name": "Bulbasaur", "HP": 45}]` → the file
```
Name,HP
Bulbasaur,45
```
You write your own version in Lab 7 (files, `with`).
**Predict:** `save_csv(rows, "test.csv")` – into which folder of your project does the file go? What is in the **first line**?


📌 **Under your nose** – exactly where to look:
- csv.DictWriter → [docs.python.org/3/library/csv.html#csv.DictWriter](https://docs.python.org/3/library/csv.html#csv.DictWriter)

In [8]:
SAVE_FOLDER = "data/raw"                     # 🔵 "data/raw", "data/processed" or "results"
FIRST_LINE = "the column names"              # 🔵 "the column names" or "the first row"


# ---------- check (do not change) ----------
check("2.7 the folder", lambda: SAVE_FOLDER.strip("/") == "data/raw", "point 1 of the description")
check("2.7 the first line", lambda: FIRST_LINE == "the column names", "point 4 of the description")

✅ 2.7 the folder
✅ 2.7 the first line


**✅ All of part 2?** Then you know what happens in the section *Lab 2* of your `main.ipynb`:
`load_csv` reads text → `convert_rows` (with `_convert`, `to_number`, `to_bool`) makes the types → `rows`. You wrote 5 of these functions yourself.

## 3 · One recursion (lecture 4) · 🧪 test data
A recursive function **calls itself** on a smaller piece of the problem, until it reaches the **stop case**:
```python
def factorial(n):
    if n == 0:                    # 1. the STOP case – without it the function never ends (RecursionError)
        return 1
    return n * factorial(n - 1)   # 2. the same problem, one step smaller
# factorial(3) = 3 * factorial(2) = 3 * 2 * factorial(1) = 3 * 2 * 1 * factorial(0) = 3 * 2 * 1 * 1 = 6
```

**🔵 Your task:** `total(values)` = the sum of a list, **with recursion** – no `sum()`, no loop.
**Recipe in words:** if the list is empty → return 0 (stop case) · otherwise → the first element + `total(` the rest of the list `)`. The rest of the list: `values[1:]`.
**You should see:** `7169` twice (the sum of the first 100 attack values).
<details><summary>💡 Hint 1 (click)</summary>

The same shape as `factorial`: `if not values: return 0`

</details>
<details><summary>💡 Hint 2 (click)</summary>

`return values[0] + total(values[1:])`

</details>

In [9]:
# ✍️ your code – total()
def total(n):
    if not n:
        return 0
    return n[0] + total(n[1:])
attacks = L.get_column(pokemon_rows, "Attack")     # 🧪
print(total(attacks[:100]), sum(attacks[:100]))


# ---------- check (do not change) ----------
check("3 total([]) = 0", lambda: total([]) == 0, "the stop case: an empty list -> 0")
check("3 total = sum", lambda: total(attacks[:100]) == 7169)

7169 7169
✅ 3 total([]) = 0
✅ 3 total = sum


## 4 · Your module `stats.py` · 🧪 test data → your file
**What it does in your project:** `loader.py` gives you `rows` → `stats.py` turns them into **numbers**: means, counts, **one number per group**.
In your block diagram it is the step **Analyse**.

**How:** **4 functions for everyone** (4.1–4.4) + **at least 2 of the menu** (4.5) – choose the ones that help **your** question.
You write and test them here; in 4.6 **you** put them into `src/stats.py`.
The tests use a mini table of 🧪 test data – the last row has missing values (`None`), like real data:

In [10]:
MINI = [                       # 🧪 TEST DATA – 6 rows
    {'name': 'Bulbasaur', 'type1': 'Grass', 'hp': 45, 'attack': 49, 'legendary': False},
    {'name': 'Ivysaur', 'type1': 'Grass', 'hp': 60, 'attack': 62, 'legendary': False},
    {'name': 'Charmander', 'type1': 'Fire', 'hp': 39, 'attack': 52, 'legendary': False},
    {'name': 'Charmeleon', 'type1': 'Fire', 'hp': 58, 'attack': 64, 'legendary': True},
    {'name': 'Squirtle', 'type1': 'Water', 'hp': 44, 'attack': 48, 'legendary': False},
    {'name': 'MissingNo', 'type1': 'Water', 'hp': None, 'attack': None, 'legendary': None},
]
print(len(MINI), "rows of test data")

6 rows of test data


### 4.1 `mean()` – the mean, with a default parameter
**Recipe:** the sum of the values / their number, then `round(…, digits)`. `digits=2` is a **default value** – like `skip_missing=True` in 2.6.
**Step by step with `[1, 2, 4]`:** `sum` → 7 · `len` → 3 · 7 / 3 → 2.3333… · `round(2.3333…, 2)` → **2.33**. With `digits=0`: `round(2.3333…, 0)` → **2.0**.
**You should see:** `2.33 2.0` and three ✅.
<details><summary>💡 Hint 1 (click)</summary>

`return round(sum(values) / len(values), digits)`

</details>

📌 **Under your nose** – exactly where to look:
- round() → [docs.python.org/3/library/functions.html#round](https://docs.python.org/3/library/functions.html#round)
- sum() / len() → [docs.python.org/3/library/functions.html#sum](https://docs.python.org/3/library/functions.html#sum)

In [11]:
def mean(values, digits=2):
    """The mean of a list of numbers, rounded to `digits` decimals.
    Example: mean([1, 2, 4]) -> 2.33  ·  mean([1, 2, 4], digits=0) -> 2.0
    """
    if not values:
        return 0.0
    return round(sum(values)/len(values), digits)


print(mean([1, 2, 4]), mean([1, 2, 4], digits=0))

# ---------- check (do not change) ----------
check("4.1 mean([1, 2, 4]) = 2.33", lambda: mean([1, 2, 4]) == 2.33)
check("4.1 mean with digits=0", lambda: mean([1, 2, 4], digits=0) == 2.0)
check("4.1 mean of hp = 49.2", lambda: mean(L.get_column(MINI, "hp")) == 49.2, "get_column skips None")

2.33 2.0
✅ 4.1 mean([1, 2, 4]) = 2.33
✅ 4.1 mean with digits=0
✅ 4.1 mean of hp = 49.2


### 4.2 `median()` – the middle value
Better than the mean for prices or salaries: one huge value does not pull it up.

**Recipe:** 1. `sorted(values)` (a **new** list) · 2. `n = len(...)` · 3. odd `n` → the element at position `n // 2` · 4. even `n` → the mean of the elements at `n // 2 - 1` and `n // 2`.
On paper: `[3, 1, 2]` → `[1, 2, 3]` → n = 3, odd → position 1 → **2** · `[4, 1, 3, 2]` → `[1, 2, 3, 4]` → n = 4, even → positions 1 and 2 → (2 + 3) / 2 = **2.5**.
Odd or even? `n % 2 == 1` means odd. **You should see:** `2 2.5` and three ✅.
<details><summary>💡 Hint 1 (click)</summary>

`ordered = sorted(values)` · `n = len(ordered)` · `if n % 2 == 1: return ordered[n // 2]`

</details>
<details><summary>💡 Hint 2 (click)</summary>

even: `return (ordered[n // 2 - 1] + ordered[n // 2]) / 2`

</details>

📌 **Under your nose** – exactly where to look:
- sorted() → [docs.python.org/3/library/functions.html#sorted](https://docs.python.org/3/library/functions.html#sorted)
- statistics.median (compare) → [docs.python.org/3/library/statistics.html#statistics.median](https://docs.python.org/3/library/statistics.html#statistics.median)

In [12]:
def median(values):
    """The middle value of a list of numbers.
    Example: median([3, 1, 2]) -> 2  ·  median([4, 1, 3, 2]) -> 2.5
    """
    ...   # 🔵 your code
    new_list = sorted(values)
    mid_value = len(new_list) // 2
    
    if len(new_list) % 2 == 1:
        return new_list[mid_value]
    else:
        return (new_list[mid_value - 1] + new_list[mid_value]) / 2

print(median([3, 1, 2]), median([4, 1, 3, 2]))

# ---------- check (do not change) ----------
check("4.2 median odd", lambda: median([3, 1, 2]) == 2)
check("4.2 median even", lambda: median([4, 1, 3, 2]) == 2.5, "the mean of the two middle values")
check("4.2 the list is not changed", lambda: (lambda v: (median(v), v)[1])([3, 1, 2]) == [3, 1, 2], "sorted(), not .sort()")

2 2.5
✅ 4.2 median odd
✅ 4.2 median even
✅ 4.2 the list is not changed


### 4.3 `count_by()` – how many of each kind
The "count per category" from Lab 3 – now as a function with a parameter.
**Recipe:** an empty dictionary · for every row: `v = row[key]` · not a key yet → put it there with 0 · add 1 · return the dictionary.
**Step by step** with 3 rows whose `type1` is `Grass`, `Grass`, `Fire`: start `{}` · row 1: `Grass` is new → `{'Grass': 0}` → +1 → `{'Grass': 1}` · row 2: `Grass` is there → `{'Grass': 2}` · row 3: `Fire` is new → `{'Grass': 2, 'Fire': 0}` → +1 → `{'Grass': 2, 'Fire': 1}`.
**You should see:** `{'Grass': 2, 'Fire': 2, 'Water': 2}` and two ✅.
<details><summary>💡 Hint 1 (click)</summary>

`counts = {}` · `for row in rows:` · `v = row[key]` · `if v not in counts: counts[v] = 0` · `counts[v] = counts[v] + 1` · `return counts`

</details>

📌 **Under your nose** – exactly where to look:
- dictionaries → [docs.python.org/3/tutorial/datastructures.html#dictionaries](https://docs.python.org/3/tutorial/datastructures.html#dictionaries)

In [13]:
def count_by(rows, key):
    """How many rows have each value of the column `key`.
    Example: count_by(rows, "type1") -> {"Grass": 2, "Fire": 2, "Water": 2}
    """
    ...   # 🔵 your code
    counts = {}
    
    for r in rows:
        value = r[key]
        if value not in counts:
            counts[value] = 0
        counts[value] = counts.get(value, 0) + 1
    return counts
print(count_by(MINI, "type1"))

# ---------- check (do not change) ----------
check("4.3 count_by type1", lambda: count_by(MINI, "type1") == {"Grass": 2, "Fire": 2, "Water": 2})
check("4.3 count_by on 800 Pokémon", lambda: count_by(pokemon_rows, "Type 1")["Water"] == 112)

{'Grass': 2, 'Fire': 2, 'Water': 2}
✅ 4.3 count_by type1
✅ 4.3 count_by on 800 Pokémon


### 4.4 `group_stat()` – one number per group
The most important function of the module: **split → calculate → collect** (Lab 3, 3.2).
1. **split:** a dictionary *group → list of values*; skip rows where the value is `None` (`continue`) · 2. **calculate:** an **inner function** `one_number(values)` – it lives inside `group_stat` because nobody else needs it, and it can read `how`:
```python
    def one_number(values):
        if how == "median":
            return median(values)
        if how == "count":
            return len(values)
        return mean(values)
```
**Your addition:** `one_number` must also understand `how="max"` → the biggest value of the group. Where do you add it?
3. **collect:** `return {k: one_number(v) for k, v in groups.items()}` – for every group (`k`) its list (`v`) becomes one number.
**Step by step with `MINI`, group = `type1`, value = `attack`:**
- split: `{'Grass': [49, 62], 'Fire': [52, 64], 'Water': [48]}` – MissingNo has `attack = None` → skipped
- calculate (`how="mean"`): Grass (49 + 62) / 2 = 55.5 · Fire (52 + 64) / 2 = 58.0 · Water 48 / 1 = 48.0
- collect: `{'Grass': 55.5, 'Fire': 58.0, 'Water': 48.0}`
**You should see:** `{'Grass': 55.5, 'Fire': 58.0, 'Water': 48.0}` and five ✅.
<details><summary>💡 Hint 1 (click)</summary>

Step 1 is like count_by – but the dictionary keeps a **list**: `groups[key] = []`, then `groups[key].append(row[value])`

</details>
<details><summary>💡 Hint 2 (click)</summary>

Copy one_number from above, indented by 4 spaces inside group_stat.

</details>

📌 **Under your nose** – exactly where to look:
- dict.items() → [docs.python.org/3/library/stdtypes.html#dict.items](https://docs.python.org/3/library/stdtypes.html#dict.items)
- dict comprehensions → [docs.python.org/3/tutorial/datastructures.html#dictionaries](https://docs.python.org/3/tutorial/datastructures.html#dictionaries)

In [14]:
def group_stat(rows, group, value, how="mean"):
    """One number per group: the mean / median / count / max of `value` in every group of `group`.
    Example: group_stat(rows, "type1", "attack") -> {"Grass": 55.5, "Fire": 58.0, "Water": 48.0}
    """
    groups = {}
    
    for r in rows:
        v = r[value]
        if v is None:
            continue
        
        g = r[group]
        if g not in groups:
            groups[g] = []
        groups[g].append(v)
    
    def one_number(values):
        if how == "mean":
            return mean(values)
        elif how == "median":
            return median(values)
        elif how == "count":
            return len(values)
        elif how == "max":
            return max(values)
        else:
            raise ValueError(f"Unknown 'how' value: {how}")

    return {g: one_number(v) for g, v in groups.items()}

print(group_stat(MINI, "type1", "attack"))

# ---------- check (do not change) ----------
check("4.4 group_stat mean", lambda: group_stat(MINI, "type1", "attack") == {"Grass": 55.5, "Fire": 58.0, "Water": 48.0})
check("4.4 group_stat median", lambda: group_stat(MINI, "type1", "attack", how="median") == {"Grass": 55.5, "Fire": 58.0, "Water": 48})
check("4.4 group_stat count (None skipped)", lambda: group_stat(MINI, "type1", "attack", how="count") == {"Grass": 2, "Fire": 2, "Water": 1})
check("4.4 group_stat max (your addition)", lambda: group_stat(MINI, "type1", "attack", how="max") == {"Grass": 62, "Fire": 64, "Water": 48}, "one more if in one_number: how == 'max' → max(values)")
check("4.4 group_stat on 800 Pokémon", lambda: group_stat(pokemon_rows, "Type 1", "Attack")["Dragon"] == 112.12)

{'Grass': 55.5, 'Fire': 58.0, 'Water': 48.0}
✅ 4.4 group_stat mean
✅ 4.4 group_stat median
✅ 4.4 group_stat count (None skipped)
✅ 4.4 group_stat max (your addition)
✅ 4.4 group_stat on 800 Pokémon


### 4.5 Menu – choose **at least 2** that help your question
Write the body of the ones you choose; leave the others with `...`. Each one is tested separately.

| Function | Use it when … |
|---|---|
| **`std(values, digits=2)`** – the spread: is your data close to the mean or all over the place? | the square root of ( the sum of (value − mean)² / (n − 1) ) |
| **`share(values, digits=2)`** – for a yes/no (`bool`) column: how often is it True? | count the True values (a counter or `sum(1 for v in values if v)`), divide by the number of values, round |
| **`filter_rows(rows, where)`** – only some rows: a function as an argument (lecture 4). | keep a row if `where(row)` is True – one line with a comprehension |
| **`top(result, n=5)`** – a ranking of the result of `group_stat` / `count_by`. | `sorted(...)` of `result.items()` by the number (`key=lambda kv: kv[1]`), from the biggest (`reverse=True`), the first `n` |
| **`min_max(values)`** – the range of a column: from … to … | a tuple `(min(...), max(...))` |
| **`summary_by(rows, group, value)`** – a small result table (like the reference project): built from **your** `group_stat`. | two calls of `group_stat` (`how="count"` and the mean) → a list of dictionaries → `sorted(..., key=lambda item: item["mean"], reverse=True)` |

**You should see:** ✅ for every function you wrote, and `chosen: …` with at least 2 names.


📌 **Under your nose** – exactly where to look:
- statistics.stdev → [docs.python.org/3/library/statistics.html#statistics.stdev](https://docs.python.org/3/library/statistics.html#statistics.stdev)
- sum() → [docs.python.org/3/library/functions.html#sum](https://docs.python.org/3/library/functions.html#sum)
- lambda → [docs.python.org/3/tutorial/controlflow.html#lambda-expressions](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions)
- sorting with key= → [docs.python.org/3/howto/sorting.html#key-functions](https://docs.python.org/3/howto/sorting.html#key-functions)
- min() / max() → [docs.python.org/3/library/functions.html#min](https://docs.python.org/3/library/functions.html#min)
- sorting with key= → [docs.python.org/3/howto/sorting.html#key-functions](https://docs.python.org/3/howto/sorting.html#key-functions)

In [15]:
def std(values, digits=2):
    """The standard deviation (sample): how far the values are from the mean.
    Example: std([2, 4, 4, 4, 5, 5, 7, 9]) -> 2.14
    """
    ...   # 🔵 your code


def share(values, digits=2):
    """The share of True values (0–1): e.g. the share of made shots.
    Example: share([True, False, True, True]) -> 0.75
    """
    ...   # 🔵 your code


def filter_rows(rows, where):
    """The rows for which where(row) is True – where is a FUNCTION (a lambda).
    Example: filter_rows(rows, lambda r: r["type1"] == "Fire") -> only the Fire rows
    """
    ...   # 🔵 your code


def top(result, n=5):
    """The n biggest (key, number) pairs of a dictionary, biggest first – a ranking.
    Example: top({"a": 1, "b": 3, "c": 2}, n=2) -> [("b", 3), ("c", 2)]
    """
    sorted_items = sorted(result.items(), key=lambda pair: pair[1], reverse=True)
    return sorted_items[:n]


def min_max(values):
    """The smallest and the biggest value as a pair.
    Example: min_max([3, 1, 2]) -> (1, 3)
    """
    ...   # 🔵 your code


def summary_by(rows, group, value):
    """A table: for every group the number of values and their mean, the biggest mean first.
    Example: summary_by(rows, "type1", "attack") -> [{"group": "Fire", "count": 2, "mean": 58.0}, ...]
    """
    counts = group_stat(rows, group, value, how="count")
    means = group_stat(rows, group, value, how="mean")
    summary = [
        {"group": g, "count": counts[g], "mean": means[g]} for g in counts]
    return sorted(summary, key=lambda x: x["mean"], reverse=True)

# ---------- check (do not change) ----------
chosen, wrong = menu_working()
for name in chosen:
    print("✅", name)
for name in wrong:
    print("❌", name, "– written, but the result is wrong – compare with the example in its description")
print("chosen:", chosen)
check("4.5 at least 2 from the menu", lambda: len(chosen) >= 2, "write the body of 2 functions you like")

✅ top
✅ summary_by
chosen: ['top', 'summary_by']
✅ 4.5 at least 2 from the menu


### 4.6 Put your functions into `src/stats.py` – yourself
1. Open your file: **📁** (left bar) → your project folder → `src` → **double-click `stats.py`** – it opens in an editor on the right.
   (For a `.py` file that is the right way. A notebook like `main.ipynb` opens differently – see 5.1.)
2. At the top there is a description from Lab 2 (`"""stats.py – Lab 3: …"""`). Leave it, or replace it with this one:
   ```python
   """stats.py – Lab 4: numbers from my data (means, counts, one number per group)."""
   ```
3. **Below it**, paste your functions from 4.1–4.4 and the ones you chose in 4.5 – **from `def` to the last line of the function**.
   **Not** the `print(...)` lines and **not** the check lines – only the functions. Keep the 4 spaces inside every `def`.
4. Save: Colab saves by itself (wait 2 s) · VS Code / PyCharm: **Ctrl + S**.
5. Run the cell below – it imports **your file** and tests it.

**You should see:** five ✅ and the names of your menu functions. ❌? Read the line: a function missing, a `...` left, a wrong indent, the file not saved.

In [16]:
require_project()
stats = fresh("stats")                   # YOUR src/stats.py, the newest version
print("functions in your stats.py:", [f for f in dir(stats) if not f.startswith("_") and callable(getattr(stats, f))])


# ---------- check (do not change) ----------
check("4.6 your file: mean", lambda: stats.mean([1, 2, 4]) == 2.33 and stats.mean([1, 2, 4], digits=0) == 2.0, "is mean() in src/stats.py and saved?")
check("4.6 your file: median", lambda: stats.median([3, 1, 2]) == 2 and stats.median([4, 1, 3, 2]) == 2.5, "is median() in src/stats.py and saved?")
check("4.6 your file: count_by", lambda: stats.count_by(MINI, "type1") == {"Grass": 2, "Fire": 2, "Water": 2}, "is count_by() in src/stats.py and saved?")
check("4.6 your file: group_stat", lambda: stats.group_stat(MINI, "type1", "attack") == {"Grass": 55.5, "Fire": 58.0, "Water": 48.0}, "is group_stat() in src/stats.py and saved?")
menu_in_file, _ = menu_working(stats)
print("menu functions in your file:", menu_in_file)
check("4.6 your file: at least 2 from the menu", lambda: len(menu_in_file) >= 2, "paste the 2 menu functions you wrote in 4.5")

functions in your stats.py: ['count_by', 'group_stat', 'mean', 'median', 'summary_by', 'top']
✅ 4.6 your file: mean
✅ 4.6 your file: median
✅ 4.6 your file: count_by
✅ 4.6 your file: group_stat
menu functions in your file: ['top', 'summary_by']
✅ 4.6 your file: at least 2 from the menu


## 5 · Your `main.ipynb` – hand in hand
### 5.1 Open `main.ipynb` the right way
⚠️ A **double-click** on `main.ipynb` in the 📁 panel of Colab opens only a **text preview** on the right – you cannot run it there.
Open it as a notebook – one of two ways:
- **From Colab:** menu **File → Open notebook** → tab **Google Drive** → type `main` in the search → click **main.ipynb** of your project.
- **From Google Drive:** drive.google.com → **My Drive** → your project folder → **right-click `main.ipynb` → Open with → Google Colaboratory**.

It opens in a new tab. **VS Code / PyCharm:** just open the file `main.ipynb` of your project.

### 5.2 What is in your `main.ipynb`
| Cell | What it does |
|---|---|
| **Setup** | connects Google Drive and finds `src/` – run it first, always |
| **Lab 2 – Data and types** | `raw = load_csv(DATA_FILE_NAME)` → text · `rows = convert_rows(raw, COLUMN_TYPES)` → types: **part 2 of today** |
| *Data overview* | only comments – hints for you |
| **Lab 3 – Functions and statistics** | `# Lab 3 – your code here` ← **today you write here** (the plan of the labs moved – the name counts, not the number) |
| Lab 4 … Lab 8 | empty – for the next labs |

**Try it:** in `main.ipynb` click **Runtime → Run all**. **You should see** under the Lab 2 cell: `rows: …` and your first row with numbers as numbers.

### 5.3 Ready examples: combining your functions (read, then adapt to your data)
Your functions are small bricks. **Combined**, they answer real questions. Below: **one question, three ways to write it**.
They use `filter_rows` and `top` from the menu (4.5) – if you did not write those two, read their description in 4.5 to understand them.

**The question:** *Which types of legendary Pokémon are the fastest?* (🧪 test data – column names `Legendary`, `Type 1`, `Speed`)

**Way 1 – line by line** (the easiest to read and to check – start like this):
```python
legendary = filter_rows(rows, lambda r: r["Legendary"])        # 1. keep only the rows you want: the legendary ones
speed_per_type = group_stat(legendary, "Type 1", "Speed")      # 2. one number per group: the mean speed of every type
best = top(speed_per_type, n=3)                                # 3. a ranking: the 3 biggest
print("fastest legendary types:", best)                        # 4. show it with a label
```
Every line makes **one** new variable – you can `print` each of them and see what happens.

**Way 2 – one line** (the same, the functions inside each other – read it **from the inside out**):
```python
print(top(group_stat(filter_rows(rows, lambda r: r["Legendary"]), "Type 1", "Speed"), n=3))
```
`filter_rows(…)` runs first → its result goes into `group_stat(…)` → that result goes into `top(…)`. Short, but harder to read and to fix.

**Way 3 – wrapped in your own function** (the steps get a name – you can use them again with other columns):
```python
def best_groups(rows, where, group, value, n=3):
    """filter -> group -> rank: the n best groups."""
    chosen = filter_rows(rows, where)
    per_group = group_stat(chosen, group, value)
    return top(per_group, n)

print(best_groups(rows, lambda r: r["Legendary"], "Type 1", "Speed"))       # the same question
print(best_groups(rows, lambda r: r["Generation"] == 1, "Type 1", "HP"))    # a new question in one line
```

**More combinations you can build:**
| Question | Combination |
|---|---|
| Which group is the biggest? | `top(count_by(rows, "GROUP"), n=3)` |
| Which group is in the middle, without outliers? | `group_stat(rows, "GROUP", "VALUE", how="median")` |
| Which group has the highest single value? | `top(group_stat(rows, "GROUP", "VALUE", how="max"), n=1)` |
| How often is it True in each group? (a yes/no column) | `group_stat(rows, "GROUP", "YES_NO_COLUMN")` – the mean of True/False = the share of True |
| The same, only for part of the data | put `filter_rows(rows, lambda r: …)` instead of `rows` |

In 5.6 you write lines like these in **your** `main.ipynb` – with **your** columns.

### 5.4 Rehearsal here – the pattern of every section of `main.ipynb` · 🧪 test data
Every analysis line in `main.ipynb` has the same 3 steps:
```python
from src.MODULE import FUNCTION              # 1. where the function comes from (your file in src/)
result = FUNCTION(rows, "COLUMN")            # 2. call it on your rows
print("what it is:", result)                 # 3. show the result – with a short label
```
Rehearse it **here** on the test data (`pokemon_rows` plays your `rows`; the functions come from part 4). Fill the gaps:
**You should see:** the number of Pokémon per type, the mean attack per type and the mean HP.
<details><summary>💡 Hint 1 (click)</summary>

`count_by(pokemon_rows, "Type 1")` · `group_stat(pokemon_rows, "Type 1", "Attack")` · `mean(L.get_column(pokemon_rows, "HP"))`

</details>

In [17]:
# 🧪 rehearsal – in main.ipynb: rows instead of pokemon_rows, and "from src.stats import …" at the top
per_type = count_by(pokemon_rows, "Type 1")
print("Pokémon per type:", per_type)

attack_per_type = group_stat(pokemon_rows, "Type 1", "Attack")
print("mean attack per type:", attack_per_type)

mean_hp = mean(L.get_column(pokemon_rows, "HP"))
print("mean HP:", mean_hp)


# ---------- check (do not change) ----------
check("5.4 count_by", lambda: per_type["Water"] == 112)
check("5.4 group_stat", lambda: attack_per_type["Dragon"] == 112.12)
check("5.4 mean + get_column", lambda: mean_hp == mean(L.get_column(pokemon_rows, "HP")))

Pokémon per type: {'Grass': 70, 'Fire': 52, 'Water': 112, 'Bug': 69, 'Normal': 98, 'Poison': 28, 'Electric': 44, 'Ground': 32, 'Fairy': 17, 'Fighting': 27, 'Psychic': 57, 'Rock': 44, 'Ghost': 32, 'Ice': 24, 'Dragon': 32, 'Dark': 31, 'Steel': 27, 'Flying': 4}
mean attack per type: {'Grass': 73.21, 'Fire': 84.77, 'Water': 74.15, 'Bug': 70.97, 'Normal': 73.47, 'Poison': 74.68, 'Electric': 69.09, 'Ground': 95.75, 'Fairy': 61.53, 'Fighting': 96.78, 'Psychic': 71.46, 'Rock': 92.86, 'Ghost': 73.78, 'Ice': 72.75, 'Dragon': 112.12, 'Dark': 88.39, 'Steel': 92.7, 'Flying': 78.75}
mean HP: 69.26
✅ 5.4 count_by
✅ 5.4 group_stat
✅ 5.4 mean + get_column


### 5.5 Your columns
Now the same with **your** data. Run the cell – it shows **your** columns and their types. Choose:
- `MY_GROUP` – a **text** column to split by (a category: city, type, month, team …),
- `MY_VALUE` – a **number** column to look at (price, temperature, points …).
Copy the names **exactly** (capital letters, spaces).

In [ ]:
# @title 👀 My columns (click ▶)
require_project()
loader = fresh("loader")
rows = loader.load_project_data()
print("✅", len(rows), "rows of YOUR data from", loader.DATA_FILE_NAME)
for col, kind in loader.COLUMN_TYPES.items():
    print(f"   {col!r:30} {kind.__name__:6}  first value: {rows[0][col]!r}")

In [ ]:
MY_GROUP = "Gender"          # 🔵 a text column of YOUR data
MY_VALUE = "CGPA"          # 🔵 a number column of YOUR data


# ---------- check (do not change) ----------
check("5.5 your columns exist", lambda: MY_GROUP in rows[0] and MY_VALUE in rows[0], "copy the names exactly from the list above")

### 5.6 Write your section in `main.ipynb`
In **your** `main.ipynb` (opened as in 5.1), find the section **"Lab 3 – Functions and statistics (`src/stats.py`)"**
and its cell `# Lab 3 – your code here`. Delete that comment and write – **by the pattern of 5.4, with your column names**:

1. **The import** – all functions you use from **your** `stats.py`:
   ```python
   from src.stats import count_by, group_stat, mean        # + your menu functions
   ```
2. **How many rows per group** – fill in your column:
   ```python
   per_group = count_by(rows, "______")
   print("rows per group:", per_group)
   ```
3. **One number per group** – now **without gaps**: `group_stat` with `MY_GROUP` and `MY_VALUE`, and a `print` with a label.
4. **One of your menu functions** on your data – your idea (e.g. `std(get_column(rows, "…"))`, `top(…)`, `summary_by(…)`) – or a combination from 5.3.
   `get_column` you can use directly: the Lab 2 cell already imported it.
5. **Runtime → Run all** in `main.ipynb`. Every cell must run without an error. ❌? Read the error in that cell – the help table at the end.

**You should see** in `main.ipynb`: your data (section Lab 2), then your numbers. **Look at them: what do they say about your question?**
<details><summary>💡 Hint 1 (click)</summary>

`means = group_stat(rows, MY_GROUP_NAME, MY_VALUE_NAME)` – in main.ipynb write the column names in quotes, e.g. `group_stat(rows, "station", "pm10")`

</details>
<details><summary>💡 Hint 2 (click)</summary>

`NameError: name 'rows' is not defined` → run the cells above first (Run all). `ImportError: cannot import name …` → that function is not in your src/stats.py (or the file is not saved).

</details>

In [ ]:
# @title 🔎 Check my main.ipynb (click ▶)
require_project()
MAIN_OK, MAIN_MSG, MAIN_USES = run_main()
print("main.ipynb uses from your stats.py:", MAIN_USES or "nothing yet")


# ---------- check (do not change) ----------
check("5.6 main.ipynb uses your stats.py", lambda: {"count_by", "group_stat"} <= set(MAIN_USES), "from src.stats import count_by, group_stat, … (step 5.6, point 1)")
check("5.6 main.ipynb runs from top to bottom", lambda: MAIN_OK, MAIN_MSG)

### 5.7 Your first finding
Two sentences, in your own words: **what do your numbers show about your question?**
Example (basketball): *"Shots at the rim go in most often (66%). Mid-range shots go in only 42% of the time."*

In [ ]:
MY_FINDING = ...   # 🔵 two sentences


# ---------- check (do not change) ----------
check("5.7 your finding", lambda: len(MY_FINDING.split()) >= 12, "two sentences, at least 12 words")

## 6 · Result check
1. Two answers **in your own words** (2–3 sentences, no AI) – the teacher may ask you to explain them.
2. Run the check. It looks at this notebook (parts 2–4), tests **your** `src/stats.py`, **runs your `main.ipynb` from top to bottom**
   and saves the report `docs/lab04_check.md` in your project.
3. Show the result to the teacher. Teams: every person commits **their own** functions (GitHub guide, part 5).

In [ ]:
EXPLAIN_LOADER = ...   # 🔵 what happens in convert_rows?
EXPLAIN_MODULE = ...   # 🔵 what does your stats.py do in your project?

In [ ]:
# @title ✅ Lab 4 result check – run it at the end
import datetime
require_project()
items = []
_words = lambda x: len(x.split()) if isinstance(x, str) else 0

# 1. parts 2–3 in this notebook
part2 = {k: v for k, v in RESULTS.items() if k[:2] in ("2.", "3 ")}
items.append((len(part2) >= 14 and all(part2.values()), "loader.py understood + recursion (parts 2–3)",
              f"{sum(part2.values())}/{len(part2)} ✅" if part2 else "run parts 2–3"))

# 2. your src/stats.py
try:
    stats = fresh("stats")
except Exception as error:
    stats = None
    print("src/stats.py:", type(error).__name__, error)
CORE = {
    "mean": lambda: stats.mean([1, 2, 4]) == 2.33 and stats.mean([1, 2, 4], digits=0) == 2.0,
    "median": lambda: stats.median([3, 1, 2]) == 2 and stats.median([4, 1, 3, 2]) == 2.5,
    "count_by": lambda: stats.count_by(MINI, "type1") == {"Grass": 2, "Fire": 2, "Water": 2},
    "group_stat": lambda: stats.group_stat(MINI, "type1", "attack") == {"Grass": 55.5, "Fire": 58.0, "Water": 48.0}
                          and stats.group_stat(MINI, "type1", "attack", how="count") == {"Grass": 2, "Fire": 2, "Water": 1},
}
core = {}
for name, test in CORE.items():
    try:
        core[name] = "✓" if test() else "✗"
    except AttributeError:
        core[name] = "✗ missing"
    except Exception as error:
        core[name] = f"✗ {type(error).__name__}"
items.append((all(v == "✓" for v in core.values()), "src/stats.py: 4 functions for everyone", ", ".join(f"{k} {v}" for k, v in core.items())))
menu_ok = menu_working(stats)[0] if stats is not None else []
items.append((len(menu_ok) >= 2, "src/stats.py: at least 2 from the menu", ", ".join(menu_ok) or "none"))

# 3. main.ipynb
MAIN_OK, MAIN_MSG, MAIN_USES = run_main()
items.append(({"count_by", "group_stat"} <= set(MAIN_USES), "main.ipynb uses your stats.py", ", ".join(MAIN_USES) or "step 5.6"))
items.append((MAIN_OK, "main.ipynb runs from top to bottom", MAIN_MSG))

# 4. finding + explanations
words_ok = _words(globals().get("MY_FINDING")) >= 12 and _words(globals().get("EXPLAIN_LOADER")) >= 10 and _words(globals().get("EXPLAIN_MODULE")) >= 10
items.append((words_ok, "your finding and explanations", "ok" if words_ok else "too short"))

score = sum(ok for ok, _, _ in items)
print(f"LAB 4 RESULT: {score}/{len(items)}\n")
for ok, name, detail in items:
    print(("✅ " if ok else "❌ ") + f"{name:46} {detail}")
lines = [f"# Lab 4 result check – {score}/{len(items)}", "", "Generated: " + datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), ""]
lines += [f"- {'✅' if ok else '❌'} {name}: {detail}" for ok, name, detail in items]
lines += ["", "## My first finding", "", str(globals().get("MY_FINDING")), "", "## Explanations", "",
          str(globals().get("EXPLAIN_LOADER")), "", str(globals().get("EXPLAIN_MODULE"))]
(PROJECT / "docs").mkdir(exist_ok=True)
(PROJECT / "docs" / "lab04_check.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\nReport saved in your project: docs/lab04_check.md")

## Help
| You see | What to do |
|---|---|
| ❌ in Setup | read the line: **what** is wrong and **which step of Lab 2** fixes it. Fix it, run Setup again |
| *This part needs your project from Lab 2* | the same – Setup found a problem; parts 2–4.5 work anyway |
| `main.ipynb` opens as text on the right | that is only a preview – open it as in **5.1** |
| ❌ in 4.6 | your `src/stats.py`: a function missing, a `...` left, a wrong indent, or the file not saved |
| `ImportError: cannot import name …` (in `main.ipynb`) | that function is not in your `src/stats.py` – or the file is not saved |
| `NameError: name 'rows' is not defined` (in `main.ipynb`) | run the cells above first: **Runtime → Run all** |
| `KeyError: '…'` | a column name: copy it **exactly** from 5.5 (capital letters, spaces) |
| *Something is not written yet* | a `...` is left or the function has no `return` |
| `TypeError: unsupported operand … NoneType` | a value is `None` – use `get_column` (it skips `None`) |
| `RecursionError` | the stop case is missing or never reached |
| `IndentationError` / `SyntaxError` | 4 spaces inside `def` / `for` / `if`; a colon `:` at the end of those lines |